# This notebook should be run with the following kernel and on wICE

`/data/leuven/software/biomed/jupyter_kernels/cbd_bioinf_v4/`

code from Kris Davie

## Notes
- I do not run this on "Substantia nigra annotated cells to save some time. I kept motor cortex, cingulate gyrus and mix"

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scarches.dataset.trvae.data_handling import remove_sparsity
import torch
import scvi
import datetime

import anndata
import scvi
import scanpy as sc

In [ ]:
sc.settings.set_figure_params(dpi=200, frameon=False)
sc.set_figure_params(dpi=200)
sc.set_figure_params(figsize=(4, 4))
torch.set_printoptions(precision=3, sci_mode=False, edgeitems=7)

scvi.settings.seed = 94705


%config InlineBackend.print_figure_kwargs={'facecolor' : "w"}
%config InlineBackend.figure_format='retina'

In [ ]:
model_dir_path = '/scANVI_Models/Public_Data/Allen_Brain_Atlas/M1/cluster_label' # change path as necessary

In [ ]:
query_adata = sc.read_h5ad(f'./hdf5/20250210_all_10x_data_cg.h5ad')

In [ ]:
query_adata

In [ ]:
query_adata.obs

In [ ]:
query_adata.layers['counts'] = query_adata.raw.X if query_adata.raw else query_adata.X

In [ ]:
query_adata.obs.region.value_counts()

In [ ]:
# Add batch column
query_adata.obs['external_donor_name_label'] = query_adata.obs['sample'].values

In [ ]:
import os

In [ ]:
# Find any procceses locking the GPU, and if it's not this one, kill it - Use carefully on multi GPU jobs
try:
    nvidia_pid = ! nvidia-smi -q | grep 'Process ID' 
    nvidia_pid = nvidia_pid[0].split(':')[1].strip()

    if os.getpid() != int(nvidia_pid):
        print(f'Killing: {nvidia_pid}')
        ! kill {nvidia_pid}
except:
    print("GPU wasn't locked")
    pass

In [ ]:
if "PCs" in query_adata.varm.keys():
    del query_adata.varm["PCs"]

In [ ]:
scvi.model.SCVI.prepare_query_anndata(
    query_adata,
    model_dir_path
)

In [ ]:
vae_q = scvi.model.SCANVI.load_query_data(
    query_adata,
    model_dir_path,
)

In [ ]:
vae_q.train(
    max_epochs=100,
    plan_kwargs=dict(weight_decay=0.0),
    check_val_every_n_epoch=10,
)

In [ ]:
query_adata.obsm["X_scANVI"] = vae_q.get_latent_representation()
query_adata.obs["predictions"] = vae_q.predict()
# The "predictions" column now contains the predicted labels

In [ ]:
index_data = query_adata.obs.index
column_data = query_adata.obs['predictions']

# Create a DataFrame with the index and column data
df = pd.DataFrame({'index': index_data, 'column_name': column_data})

date = datetime.datetime.now().strftime('%Y%m%d')
# Write the DataFrame to a CSV file
df.to_csv(f'scANVI/{date}_10x_all_celltype_fine_annotation_from_scANVI_cg.csv', index=False)

# Subclass label model

In [ ]:
model_dir_path = 'scANVI_Models/Public_Data/Allen_Brain_Atlas/M1/subclass_label' # Entry from "Data Location" column

In [ ]:
query_adata = sc.read_h5ad(f'./hdf5/20250210_all_10x_data_cg.h5ad')

In [ ]:
query_adata.layers['counts'] = query_adata.raw.X if query_adata.raw else query_adata.X

In [ ]:
query_adata = query_adata[(query_adata.obs.region != 'Substantia Nigra')] 

In [ ]:
query_adata

In [ ]:
# Add batch column
query_adata.obs['external_donor_name_label'] = query_adata.obs['sample'].values

In [ ]:
if "PCs" in query_adata.varm.keys():
    del query_adata.varm["PCs"]

In [ ]:
scvi.model.SCVI.prepare_query_anndata(
    query_adata,
    model_dir_path
)

In [ ]:
vae_q = scvi.model.SCANVI.load_query_data(
    query_adata,
    model_dir_path,
)

In [ ]:
vae_q.train(
    max_epochs=100,
    plan_kwargs=dict(weight_decay=0.0),
    check_val_every_n_epoch=10,
)

In [ ]:
query_adata.obsm["X_scANVI"] = vae_q.get_latent_representation()
query_adata.obs["predictions"] = vae_q.predict()
# The "predictions" column now contains the predicted labels

In [ ]:
index_data = query_adata.obs.index
column_data = query_adata.obs['predictions']

# Create a DataFrame with the index and column data
df = pd.DataFrame({'index': index_data, 'column_name': column_data})


# Write the DataFrame to a CSV file
df.to_csv(f'scANVI/{date}_10x_all_subclusters_from_scANVI_cg.csv', index=False)

In [ ]:
pip list

```
Package                      Version
---------------------------- --------------
absl-py                      1.2.0
aenum                        3.1.11
aioconsole                   0.5.0
aiohttp                      3.8.3
aiosignal                    1.2.0
albumentations               0.4.6
anndata                      0.8.0
annoy                        1.17.1
appdirs                      1.4.4
arboreto                     0.1.6
arrow                        1.2.1
asciitree                    0.3.3
asttokens                    2.0.8
astunparse                   1.6.3
async-timeout                4.0.2
attrs                        22.1.0
backcall                     0.2.0
backports.zoneinfo           0.2.1
batchglm                     0.7.4
bbknn                        1.5.1
beautifulsoup4               4.11.1
bhtsne                       0.1.9
binaryornot                  0.4.4
black                        22.1.0
bleach                       5.0.1
bokeh                        2.3.3
cachetools                   5.2.0
cachy                        0.3.0
certifi                      2022.9.24
cffi                         1.15.1
chardet                      5.0.0
charset-normalizer           2.0.7
chex                         0.1.5
cleo                         0.8.1
click                        8.0.4
clikit                       0.6.2
cloudpickle                  2.2.0
cmake                        3.24.1.1
colorcet                     2.0.6
commonmark                   0.9.1
cookiecutter                 1.7.3
crashtest                    0.3.1
crcmod                       1.7
cryptography                 38.0.1
cycler                       0.11.0
Cython                       0.29.32
dask                         2022.2.0
dask-image                   2022.9.0
datashader                   0.14.1rc1
datashape                    0.5.2
debugpy                      1.6.3
decorator                    5.1.1
defusedxml                   0.7.1
Deprecated                   1.2.13
diffxpy                      0.7.4
dill                         0.3.5.1
distlib                      0.3.1
distributed                  2022.2.0
dm-tree                      0.1.7
docker-pycreds               0.4.0
docrep                       0.3.2
entrypoints                  0.4
et-xmlfile                   1.1.0
etils                        0.8.0
executing                    1.1.1
fa2                          0.3.5
fast-enum                    1.3.0
fasteners                    0.18
fbpca                        1.0
fcsparser                    0.2.4
filelock                     3.0.12
flatbuffers                  2.0.7
flax                         0.6.1
FlowCytometryTools           0.5.1
fonttools                    4.37.4
frozenlist                   1.3.1
fsspec                       2022.8.2
future                       0.18.2
gast                         0.4.0
gdown                        4.5.1
gefpy                        0.6.3
geosketch                    1.2
gitdb                        4.0.9
GitPython                    3.1.29
glog                         0.3.1
google-auth                  2.12.0
google-auth-oauthlib         0.4.6
google-pasta                 0.2.0
gprofiler-official           1.0.0
graphtools                   1.5.2
grpcio                       1.49.1
gtfparse                     1.2.1
h5py                         3.7.0
harmonypy                    0.0.6
harmonyTS                    0.1.4
HeapDict                     1.0.1
holoviews                    1.14.5
hotspotsc                    0.9.1
html5lib                     1.1
hvplot                       0.7.3
idna                         3.4
igraph                       0.9.10
imageio                      2.9.0
imgaug                       0.4.0
importlib-metadata           5.0.0
importlib-resources          5.10.0
inflect                      6.0.0
intervaltree                 3.1.0
ipykernel                    6.16.0
ipython                      8.5.0
ipywidgets                   8.0.2
jax                          0.3.21
jaxlib                       0.3.20
jedi                         0.18.1
jeepney                      0.6.0
Jinja2                       3.0.3
jinja2-time                  0.2.0
joblib                       1.0.1
jsonschema                   4.16.0
jupyter_client               7.3.5
jupyter-core                 4.11.1
jupyterlab-widgets           3.0.3
KDEpy                        1.1.0
keras                        2.7.0
Keras-Preprocessing          1.1.2
keyring                      21.8.0
kiwisolver                   1.4.4
leidenalg                    0.8.10
libclang                     14.0.6
llvmlite                     0.39.1
locket                       1.0.0
loompy                       3.0.7
louvain                      0.7.1
magic-impute                 3.0.0
Markdown                     3.4.1
MarkupSafe                   2.1.1
matplotlib                   3.5.2
matplotlib-inline            0.1.6
matplotlib-scalebar          0.8.1
milopy                       0.1.1
mnnpy                        0.1.9.5
msgpack                      1.0.2
mudata                       0.2.0
MulticoreTSNE                0.1
multidict                    6.0.2
multipledispatch             0.6.0
mypy-extensions              0.4.3
natsort                      7.1.1
nest-asyncio                 1.5.6
networkx                     2.8.7
newick                       1.0.0
numba                        0.56.2
numcodecs                    0.10.2
numexpr                      2.8.3
numpy                        1.21.4
numpy-groupies               0.9.19
numpyro                      0.10.1
oauthlib                     3.2.1
omnipath                     1.0.5
opencv-python                4.5.5.64
opencv-python-headless       4.6.0.66
openpyxl                     3.0.10
opt-einsum                   3.3.0
optax                        0.1.3
packaging                    21.3
palantir                     1.0.1
pandas                       1.3.4
panel                        0.12.1
param                        1.11.1
parso                        0.8.3
partd                        1.3.0
pastel                       0.2.1
pathspec                     0.9.0
pathtools                    0.1.2
patsy                        0.5.3
pbr                          5.5.1
pexpect                      4.8.0
PhenoGraph                   1.5.7
pickleshare                  0.7.5
Pillow                       9.2.0
PIMS                         0.6.1
pip                          22.3
pkginfo                      1.6.1
pkgutil_resolve_name         1.3.10
platformdirs                 2.5.1
poetry-core                  1.0.7
poyo                         0.5.0
promise                      2.3
prompt-toolkit               3.0.31
protobuf                     3.19.4
psutil                       5.9.2
ptyprocess                   0.7.0
pure-eval                    0.2.2
pyarrow                      8.0.0
pyasn1                       0.4.8
pyasn1-modules               0.2.8
pycparser                    2.21
pyct                         0.4.8
pydantic                     1.10.2
pyDeprecate                  0.3.2
Pygments                     2.13.0
PyGSP                        0.5.1
pylev                        1.3.0
pymde                        0.1.15
pynndescent                  0.5.7
pyparsing                    3.0.9
pyro-api                     0.1.2
pyro-ppl                     1.8.0
pyrsistent                   0.18.1
PySocks                      1.7.1
python-bps-continued         7
python-dateutil              2.8.2
python-gflags                3.1.2
python-slugify               5.0.2
python-snappy                0.6.1
pytorch-lightning            1.7.7
pytz                         2022.4
pytz-deprecation-shim        0.1.0.post0
pyviz-comms                  2.2.1
PyWavelets                   1.4.1
PyYAML                       6.0
pyzmq                        24.0.1
requests                     2.26.0
requests-oauthlib            1.3.1
requests-toolbelt            0.9.1
retrying                     1.3.3
rich                         12.6.0
rpy2                         3.5.5
rsa                          4.9
scanorama                    1.7.2
scanpy                       1.9.1
scArches                     0.5.4
scHPL                        1.0.1
scikit-image                 0.19.0
scikit-learn                 1.0.1
scikit-misc                  0.1.4
scipy                        1.7.3
scprep                       1.2.1
scrublet                     0.2.3
scvi-tools                   0.17.4
seaborn                      0.11.2
SecretStorage                3.3.0
sentry-sdk                   1.9.0
session-info                 1.0.0
setproctitle                 1.3.2
setuptools                   45.2.0
Shapely                      1.7.1
shellingham                  1.3.2
shortuuid                    1.0.9
six                          1.16.0
sklearn                      0.0
slicerator                   1.1.0
slideio                      0.5.225
smmap                        5.0.0
sortedcontainers             2.4.0
soupsieve                    2.3.2.post1
sparse                       0.13.0
spatialpandas                0.4.4
squidpy                      1.2.2
stack-data                   0.5.1
statsmodels                  0.12.1
stdlib-list                  0.8.0
stereopy                     0.6.0
tables                       3.7.0
tasklogger                   1.2.0
tblib                        1.7.0
tensorboard                  2.10.1
tensorboard-data-server      0.6.1
tensorboard-plugin-wit       1.8.1
tensorflow                   2.7.0
tensorflow-estimator         2.7.0
tensorflow-io-gcs-filesystem 0.24.0
termcolor                    2.0.1
testresources                2.0.1
text-unidecode               1.3
texttable                    1.6.4
threadpoolctl                3.1.0
tifffile                     2021.11.2
tomli                        2.0.1
tomlkit                      0.7.0
toolz                        0.12.0
torch                        1.10.0+cu111
torchaudio                   0.10.0+rocm4.1
torchmetrics                 0.10.0
torchvision                  0.11.1+cu111
tornado                      6.2
tqdm                         4.60.0
traitlets                    5.4.0
typing_extensions            4.3.0
tzdata                       2022.4
tzlocal                      4.2
umap-learn                   0.5.1
urllib3                      1.26.9
validators                   0.20.0
virtualenv                   20.2.2
wandb                        0.13.4
wcwidth                      0.2.5
webencodings                 0.5.1
websockets                   10.3
Werkzeug                     2.2.2
wheel                        0.34.2
widgetsnbextension           4.0.3
wishbone-dev                 0.5.2
wrapt                        1.14.1
xarray                       0.20.1
xlrd                         2.0.1
yarl                         1.8.1
zarr                         2.13.3
zict                         2.2.0
zipp                         3.9.0
```